In [1]:
import os
import pandas as pd
import numpy as np
import boto3
import datetime as dt
from tqdm import tqdm

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-11-26 14:36:45.392567


#### Functions

In [3]:
def get_list_of_files_in_s3_location(cls_client, str_bucket_name, str_prefix):
    list_str_filename = []
    continuation_token = None

    while True:
        if continuation_token:
            dict_response = cls_client.list_objects_v2(
                Bucket=str_bucket_name,
                Prefix=str_prefix,
                ContinuationToken=continuation_token
            )
        else:
            dict_response = cls_client.list_objects_v2(
                Bucket=str_bucket_name,
                Prefix=str_prefix
            )

        # Add the filenames from this batch
        list_dict_contents = dict_response.get('Contents', [])
        for dict_contents in list_dict_contents:
            file_key = dict_contents['Key']
            if 'gzip' in file_key:
                list_str_filename.append(file_key)

        # Check if there are more files to fetch
        continuation_token = dict_response.get('NextContinuationToken')
        if not continuation_token:
            break

    return list_str_filename

#### Constants

In [4]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# output
str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 03_get_requests


#### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

#### Get the funded accounts

In [6]:
%%time

str_uri = f's3://{str_project}/02_get_tsp_data/df_tsp.gzip'
list_account_funded = list(pd.read_parquet(str_uri)['bigaccountid__app'])
print(f'There are {len(list_account_funded)} funded accounts')
print()

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


There are 99681 funded accounts

CPU times: user 313 ms, sys: 88.1 ms, total: 401 ms
Wall time: 519 ms


#### Get list of files - tblDove

In [7]:
cls_client = boto3.client('s3')
list_str_filename_tbldove = get_list_of_files_in_s3_location(
    cls_client=cls_client,
    str_bucket_name='20241022-parse-snowflake-payloads',
    str_prefix='deprecated/03_pull_payloads_tbldove',
)
print(f'There are {len(list_str_filename_tbldove)} days from tblDove')

There are 855 days from tblDove


#### Get list of files - Snowflake

In [8]:
cls_client = boto3.client('s3')
list_str_filename_snowflake = get_list_of_files_in_s3_location(
    cls_client=cls_client,
    str_bucket_name='20241022-parse-snowflake-payloads',
    str_prefix='01_pull_payloads',
)
print(f'There are {len(list_str_filename_snowflake)} days from snowflake')

There are 1147 days from snowflake


#### Combine lists

In [9]:
list_str_filekey = list_str_filename_tbldove + list_str_filename_snowflake
print(f'There are {len(list_str_filekey)} total files')

There are 2002 total files


#### Reverse the list

In [10]:
list_str_filekey.reverse()

#### Iterate through files

In [ ]:
list_accountid_gethered = []
list_df = []
for str_filekey in tqdm(list_str_filekey):
    #print(str_filekey)
    
    # bool for tbldove
    if 'deprecated' in str_filekey:
        bool_tbldove = True
    else:
        bool_tbldove = False
    
    # create uri
    str_uri = f's3://20241022-parse-snowflake-payloads/{str_filekey}'
    # import
    df = pd.read_parquet(str_uri)
    
    # rename
    dict_rename = {
        'bigAccountId': 'ACCOUNTID',
        'dtmCreatedDate': 'REQUEST_DATETIME',
        'strRequest': 'REQUEST_JSON',
    }
    df.rename(columns=dict_rename, inplace=True)
    
    # assign
    if bool_tbldove: 
        df['RESPONSE_MODEL_NAME'] = 'Gen10'
    else:
        pass
    
    # set accountid to int
    df['ACCOUNTID'] = df['ACCOUNTID'].astype(int)
    
    # subset to only funded
    df = df[df['ACCOUNTID'].isin(list_account_funded)].copy()
    
    # only certain models (i.e., rm dlv1)
    list_str_model = [
        'Gen10',
        'PRESTIGE-GENXI',
        'PRESTIGE-GEN-XII',
    ]
    df = df[df['RESPONSE_MODEL_NAME'].isin(list_str_model)].copy()
    
    # get unique values for model name
    list_str_models_unique = list(df['RESPONSE_MODEL_NAME'].value_counts().index)
    # if there are gen 11 payloads
    if 'PRESTIGE-GEN-XII' in list_str_models_unique:
        # use only gen 12
        df = df[df['RESPONSE_MODEL_NAME'] == 'PRESTIGE-GEN-XII'].copy()
    else:
        pass

    # get last request for the account
    df.sort_values(by='REQUEST_DATETIME', ascending=True, inplace=True)
    df.drop_duplicates(subset=['ACCOUNTID'], keep='last', inplace=True)
    
    # make sure the account has not already been gathered
    df = df[~df['ACCOUNTID'].isin(list_accountid_gethered)].copy()
    
    # get list of ID
    list_id_tmp = list(df['ACCOUNTID'])
    # extend
    list_accountid_gethered += list_id_tmp
    
    # rm dups
    list_accountid_gethered = list(dict.fromkeys(list_accountid_gethered))
    
    # make filekey
    df['FILE_KEY'] = str_filekey
    
    # subset
    list_cols = [
        'ACCOUNTID',
        'REQUEST_DATETIME',
        'REQUEST_JSON',
        'RESPONSE_MODEL_NAME',
        'FILE_KEY',
    ]
    df = df[list_cols].copy()
    
    # append
    list_df.append(df)

100%|██████████| 2002/2002 [4:46:06<00:00,  8.57s/it]  


#### Make df

In [ ]:
%%time

df = pd.concat(list_df)
# show
df

CPU times: user 300 ms, sys: 3.65 ms, total: 303 ms
Wall time: 303 ms


,ACCOUNTID,REQUEST_DATETIME,REQUEST_JSON,RESPONSE_MODEL_NAME,FILE_KEY
2797,8403955,2024-11-25 21:28:42+00:00,"{\n ""request_id"": ""8403955631382"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
5533,8402005,2024-11-25 21:36:27+00:00,"{\n ""request_id"": ""8402005630839"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
3934,8413195,2024-11-25 21:47:20+00:00,"{\n ""request_id"": ""841319583975"",\n ""rows"": ...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
7757,8408002,2024-11-25 21:57:42+00:00,"{\n ""request_id"": ""8408002113072"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
2213,8345820,2024-11-25 22:08:51+00:00,"{\n ""request_id"": ""8345820685224"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
...,...,...,...,...,...
1146,5703249,2021-07-27 16:46:07.2159729,"{""request_id"":""590120"",""rows"":[{""row_id"":""5703...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...
1198,5704330,2021-07-27 17:18:03.4670093,"{""request_id"":""590172"",""rows"":[{""row_id"":""5704...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...
41,5702434,2021-07-26 16:29:29.3903686,"{""request_id"":""588736"",""rows"":[{""row_id"":""5702...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...
61,5714239,2021-07-26 16:39:34.1121025,"{""request_id"":""588761"",""rows"":[{""row_id"":""5714...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...


#### Convert non-numeric to string

In [17]:
for col in tqdm(df.columns):
    # get dtype
    str_dtype = df[col].dtype
    # logic
    if str_dtype not in ['int64','float64']:
        # convert to str
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 5/5 [00:00<00:00, 23.20it/s]


#### Write to s3

In [18]:
%%time

str_filename = 'df_requests.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 5min, sys: 32.1 s, total: 5min 32s
Wall time: 6min 11s
